# HSGP Residuals on GAM and Huber Means

This notebook tests whether additive Hilbert-space GP residual corrections improve the tuned GAM and Huber signed-delta mean models. It keeps the residual model small and diagnostic: fit the mean model first, then fit a Student-t MAP model on additive Hilbert basis residual features.

## 1. Setup

In [1]:
"Configure imports, paths, and deterministic settings."

from datetime import date
from pathlib import Path
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = next(parent for parent in Path.cwd().parents if (parent / "src").exists())

import sys
sys.path.append(str(ROOT))

from src.utils.run_bundle import RunBundle

RUN_DATE = "2026-06-22"
RUN_TAG = "bayesian_hsgp_residual_on_gam_huber"
SEED = 42
N_SPLITS = 3
MAX_TARGET_ROWS_PER_WELL = 140
MAX_TRAIN_ROWS_PER_FOLD = 80000
RESIDUAL_SAMPLE_PER_FOLD = 1800

NU = 3.0
HSGP_M = 8
HSGP_ELL = 3.0
HSGP_PRIOR_SCALE = 0.55
INTERCEPT_PRIOR_SCALE = 1.0
SIGMA_PRIOR_SCALE = 1.5
MAX_OPT_ITER = 900

GAM_N_KNOTS = 7
GAM_DEGREE = 3
GAM_ALPHA = 60.0
HUBER_ALPHA = 0.0001
HUBER_EPSILON = 1.35

DATA_DIR = ROOT / "data" / "raw"
TRAIN_DIR = DATA_DIR / "train"
ARTIFACT_DIR = ROOT / "artifacts"
TABLE_DIR = ARTIFACT_DIR / "tables"
FIGURE_DIR = ARTIFACT_DIR / "figures"
REPORT_DIR = ARTIFACT_DIR / "reports"
for directory in [TABLE_DIR, FIGURE_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SMOOTH_FEATURES = [
    "distance_from_boundary_md",
    "eval_progress",
    "MD",
    "Z",
    "md_norm",
    "dX_from_boundary",
    "dY_from_boundary",
    "dZ_from_boundary",
    "horizontal_dist_from_boundary",
    "tvt_slope_last_10",
    "tvt_slope_last_25",
    "tvt_slope_last_50",
    "tvt_slope_last_100",
    "z_slope_last_25",
    "z_slope_last_50",
    "z_slope_last_100",
    "tvt_slope_last_10_projected_delta",
    "tvt_slope_last_25_projected_delta",
    "tvt_slope_last_50_projected_delta",
    "tvt_slope_last_100_projected_delta",
]
LINEAR_FEATURES = ["last_known_tvt"]
FEATURE_COLUMNS = list(dict.fromkeys(SMOOTH_FEATURES + LINEAR_FEATURES))
HSGP_COORDINATES = ["distance_log1p", "eval_progress", "abs_projected_delta_10"]

rng = np.random.default_rng(SEED)

print({"root": str(ROOT), "run_tag": RUN_TAG, "date": RUN_DATE})

{'root': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling', 'run_tag': 'bayesian_hsgp_residual_on_gam_huber', 'date': '2026-06-22'}


## 2. Feature Construction

In [2]:
"Define notebook-local slope-core feature construction and grouped folds."


def discover_wells(directory):
    return sorted(path.name.replace("__horizontal_well.csv", "") for path in Path(directory).glob("*__horizontal_well.csv"))


def load_horizontal_well(path):
    df = pd.read_csv(path)
    df.index.name = "row_idx"
    df["row_idx"] = df.index
    return df


def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce").astype(float)


def deterministic_sample_positions(mask, max_rows):
    positions = np.flatnonzero(mask.to_numpy())
    if len(positions) <= max_rows:
        return positions
    return positions[np.linspace(0, len(positions) - 1, max_rows).astype(int)]


def deterministic_row_cap(frame, max_rows):
    if len(frame) <= max_rows:
        return frame
    positions = np.linspace(0, len(frame) - 1, max_rows).astype(int)
    return frame.iloc[positions].copy()


def window_slope(md, values, end_pos, window):
    start_pos = max(0, end_pos - window + 1)
    x0 = float(md.iloc[start_pos])
    x1 = float(md.iloc[end_pos])
    y0 = float(values.iloc[start_pos])
    y1 = float(values.iloc[end_pos])
    if not np.isfinite([x0, x1, y0, y1]).all() or abs(x1 - x0) < 1e-9:
        return np.nan
    return (y1 - y0) / (x1 - x0)


def build_slope_core_features_for_well(well, directory, max_train_rows=None):
    horizontal = load_horizontal_well(directory / f"{well}__horizontal_well.csv")
    known_mask = horizontal["TVT_input"].notna()
    target_mask = horizontal["TVT_input"].isna() & horizontal["TVT"].notna()
    if int(target_mask.sum()) == 0 or int(known_mask.sum()) == 0:
        return pd.DataFrame()

    tvt_input = safe_numeric(horizontal["TVT_input"])
    md = safe_numeric(horizontal["MD"])
    x = safe_numeric(horizontal["X"])
    y = safe_numeric(horizontal["Y"])
    z = safe_numeric(horizontal["Z"])

    known_positions = np.flatnonzero(known_mask.to_numpy())
    last_known_pos = int(known_positions[-1])
    last_known_tvt = float(tvt_input.iloc[last_known_pos])
    last_known_md = float(md.iloc[last_known_pos])
    last_known_x = float(x.iloc[last_known_pos])
    last_known_y = float(y.iloc[last_known_pos])
    last_known_z = float(z.iloc[last_known_pos])

    slope_windows = [10, 25, 50, 100]
    tvt_slopes = {f"tvt_slope_last_{window}": window_slope(md, tvt_input, last_known_pos, window) for window in slope_windows}
    z_slopes = {f"z_slope_last_{window}": window_slope(md, z, last_known_pos, window) for window in slope_windows}
    horizontal_step = np.sqrt(x.diff().fillna(0.0) ** 2 + y.diff().fillna(0.0) ** 2)
    horizontal_dist = horizontal_step.cumsum()

    target_positions = deterministic_sample_positions(target_mask, max_train_rows) if max_train_rows is not None else np.flatnonzero(target_mask.to_numpy())
    all_target_positions = np.flatnonzero(target_mask.to_numpy())
    eval_rank = {int(pos): rank for rank, pos in enumerate(all_target_positions)}
    eval_denominator = max(len(all_target_positions) - 1, 1)

    rows = []
    for pos in target_positions:
        distance_md = float(md.iloc[pos] - last_known_md)
        tvt_slope_10 = tvt_slopes["tvt_slope_last_10"]
        projected_delta_10 = tvt_slope_10 * distance_md if np.isfinite(tvt_slope_10) else np.nan
        row = {
            "wellname": well,
            "row_idx": int(horizontal.index[pos]),
            "TVT": float(horizontal["TVT"].iloc[pos]),
            "last_known_tvt": last_known_tvt,
            "MD": float(md.iloc[pos]),
            "Z": float(z.iloc[pos]),
            "delta_tvt_from_boundary": float(horizontal["TVT"].iloc[pos]) - last_known_tvt,
            "md_norm": float((md.iloc[pos] - md.min()) / (md.max() - md.min() + 1e-9)),
            "distance_from_boundary_md": distance_md,
            "distance_log1p": float(np.log1p(max(distance_md, 0.0))),
            "eval_progress": float(eval_rank[int(pos)] / eval_denominator),
            "dX_from_boundary": float(x.iloc[pos] - last_known_x),
            "dY_from_boundary": float(y.iloc[pos] - last_known_y),
            "dZ_from_boundary": float(z.iloc[pos] - last_known_z),
            "horizontal_dist_from_boundary": float(horizontal_dist.iloc[pos] - horizontal_dist.iloc[last_known_pos]),
            "abs_projected_delta_10": float(abs(projected_delta_10)) if np.isfinite(projected_delta_10) else np.nan,
        }
        for name, value in tvt_slopes.items():
            row[name] = float(value) if np.isfinite(value) else np.nan
            row[f"{name}_projected_delta"] = row[name] * distance_md if np.isfinite(row[name]) else np.nan
        for name, value in z_slopes.items():
            row[name] = float(value) if np.isfinite(value) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def group_well_folds(wells, n_splits):
    ordered = sorted(wells)
    valid_groups = [list(group) for group in np.array_split(ordered, min(n_splits, len(ordered)))]
    folds = []
    for valid in valid_groups:
        valid_set = set(valid)
        train = [well for well in ordered if well not in valid_set]
        folds.append((train, valid))
    return folds


def metric_row(y_true, y_pred):
    return {
        "rmse": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }

In [3]:
"Build the sampled target-zone frame and additive residual coordinates."

frames = []
for well in discover_wells(TRAIN_DIR):
    frame = build_slope_core_features_for_well(well, TRAIN_DIR, max_train_rows=MAX_TARGET_ROWS_PER_WELL)
    if not frame.empty:
        frames.append(frame)

feature_frame = pd.concat(frames, ignore_index=True)

print({"wells": int(feature_frame["wellname"].nunique()), "rows": int(len(feature_frame)), "target_mean": float(feature_frame["TVT"].mean())})
feature_frame[["wellname", "row_idx", "TVT", "delta_tvt_from_boundary"] + HSGP_COORDINATES].head()

{'wells': 773, 'rows': 108220, 'target_mean': 11552.472777582701}


,wellname,row_idx,TVT,delta_tvt_from_boundary,distance_log1p,eval_progress,abs_projected_delta_10
0,000d7d20,1442,11747.38,0.01,0.693147,0.000000,0.011111
1,000d7d20,1469,11747.52,0.15,3.367296,0.007040,0.311111
2,000d7d20,1497,11747.46,0.09,4.043051,0.014342,0.622222
3,000d7d20,1524,11748.05,0.68,4.430817,0.021382,0.922222
4,000d7d20,1552,11748.66,1.29,4.718499,0.028683,1.233333


## 3. Mean Models And Residual HSGP

In [4]:
"Define mean models and additive Hilbert residual MAP helpers."


def make_gam_model():
    transformer = ColumnTransformer(
        transformers=[
            ("smooth", Pipeline([("imputer", SimpleImputer(strategy="median")), ("spline", SplineTransformer(n_knots=GAM_N_KNOTS, degree=GAM_DEGREE, include_bias=False)), ("scale", StandardScaler())]), SMOOTH_FEATURES),
            ("linear", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), LINEAR_FEATURES),
        ],
        remainder="drop",
    )
    return Pipeline([("features", transformer), ("model", Ridge(alpha=GAM_ALPHA))])


def make_huber_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("model", HuberRegressor(alpha=HUBER_ALPHA, epsilon=HUBER_EPSILON, max_iter=1200)),
    ])


def fit_coord_preprocessor(frame, columns):
    medians = frame[columns].median(numeric_only=True)
    filled = frame[columns].fillna(medians)
    means = filled.mean()
    stds = filled.std().replace(0.0, 1.0).fillna(1.0)
    return medians, means, stds


def apply_coord_preprocessor(frame, columns, medians, means, stds):
    filled = frame[columns].fillna(medians)
    return ((filled - means) / stds).to_numpy(float)


def hilbert_basis_1d(values, m=HSGP_M, ell=HSGP_ELL):
    clipped = np.clip(values, -ell, ell)
    shifted = (clipped + ell) / (2.0 * ell)
    return np.column_stack([np.sqrt(1.0 / ell) * np.sin(np.pi * j * shifted) for j in range(1, m + 1)])


def additive_hilbert_design(frame, model, coordinates):
    coords = apply_coord_preprocessor(frame, coordinates, model["medians"], model["means"], model["stds"])
    blocks = [hilbert_basis_1d(coords[:, idx]) for idx in range(coords.shape[1])]
    return np.column_stack(blocks)


def softplus(value):
    return np.logaddexp(0.0, value)


def residual_student_t_objective(params, X, y):
    intercept = params[0]
    beta = params[1:-1]
    sigma = softplus(params[-1]) + 1e-6
    residual = y - (intercept + X @ beta)
    scaled = residual / sigma
    nll = np.sum(np.log(sigma) + 0.5 * (NU + 1.0) * np.log1p((scaled ** 2) / NU))
    beta_prior = 0.5 * np.sum((beta / HSGP_PRIOR_SCALE) ** 2)
    intercept_prior = 0.5 * (intercept / INTERCEPT_PRIOR_SCALE) ** 2
    sigma_prior = 0.5 * (sigma / SIGMA_PRIOR_SCALE) ** 2
    return float(nll + beta_prior + intercept_prior + sigma_prior)


def fit_hsgp_residual_model(train_frame, residual, coordinates):
    medians, means, stds = fit_coord_preprocessor(train_frame, coordinates)
    model = {"coordinates": coordinates, "medians": medians, "means": means, "stds": stds}
    X = additive_hilbert_design(train_frame, model, coordinates)
    y_mean = float(np.mean(residual))
    y_std = float(np.std(residual) + 1e-9)
    y = (residual - y_mean) / y_std
    start = np.zeros(X.shape[1] + 2, dtype=float)
    start[-1] = np.log(np.expm1(1.0))
    result = minimize(
        residual_student_t_objective,
        start,
        args=(X, y),
        method="L-BFGS-B",
        options={"maxiter": MAX_OPT_ITER, "ftol": 1e-8, "maxls": 50},
    )
    if not result.success:
        print({"optimizer_warning": str(result.message), "coordinates": coordinates})
    model.update({"params": result.x, "y_mean": y_mean, "y_std": y_std, "loss": float(result.fun), "success": bool(result.success), "nit": int(result.nit), "n_basis": int(X.shape[1])})
    return model


def predict_hsgp_residual(model, frame):
    X = additive_hilbert_design(frame, model, model["coordinates"])
    params = model["params"]
    pred_scaled = params[0] + X @ params[1:-1]
    return pred_scaled * model["y_std"] + model["y_mean"]


candidate_specs = [
    {"candidate": "gam_mean_only", "mean": "gam", "coordinates": []},
    {"candidate": "gam_plus_hsgp_dp_m8", "mean": "gam", "coordinates": ["distance_log1p", "eval_progress"]},
    {"candidate": "gam_plus_hsgp_dps_m8", "mean": "gam", "coordinates": ["distance_log1p", "eval_progress", "abs_projected_delta_10"]},
    {"candidate": "huber_mean_only", "mean": "huber", "coordinates": []},
    {"candidate": "huber_plus_hsgp_dp_m8", "mean": "huber", "coordinates": ["distance_log1p", "eval_progress"]},
    {"candidate": "huber_plus_hsgp_dps_m8", "mean": "huber", "coordinates": ["distance_log1p", "eval_progress", "abs_projected_delta_10"]},
]

## 4. Grouped CV Sweep

In [5]:
"Run grouped well-fold validation for mean-only and residual-HSGP candidates."

fit_start = time.time()
fold_rows = []
residual_rows = []
posterior_rows = []
wells = sorted(feature_frame["wellname"].unique())
folds = group_well_folds(wells, N_SPLITS)

for fold_idx, (train_fold_wells, valid_fold_wells) in enumerate(folds, start=1):
    train_frame = feature_frame.loc[feature_frame["wellname"].isin(train_fold_wells)].copy()
    valid_frame = feature_frame.loc[feature_frame["wellname"].isin(valid_fold_wells)].copy()
    train_frame = deterministic_row_cap(train_frame, MAX_TRAIN_ROWS_PER_FOLD)
    y_train_delta = train_frame["delta_tvt_from_boundary"].to_numpy(float)
    y_valid_tvt = valid_frame["TVT"].to_numpy(float)
    valid_last_known = valid_frame["last_known_tvt"].to_numpy(float)

    mean_models = {}
    for mean_name in ["gam", "huber"]:
        if mean_name == "gam":
            model = make_gam_model()
        else:
            model = make_huber_model()
        model.fit(train_frame[FEATURE_COLUMNS], y_train_delta)
        mean_models[mean_name] = model

    for spec in candidate_specs:
        mean_model = mean_models[spec["mean"]]
        train_mean_delta = mean_model.predict(train_frame[FEATURE_COLUMNS])
        valid_mean_delta = mean_model.predict(valid_frame[FEATURE_COLUMNS])
        residual_model = None
        residual_correction = np.zeros(len(valid_frame), dtype=float)
        final_loss = np.nan
        optimizer_success = True
        optimizer_iterations = 0
        n_basis = 0

        if spec["coordinates"]:
            train_residual = y_train_delta - train_mean_delta
            residual_model = fit_hsgp_residual_model(train_frame, train_residual, spec["coordinates"])
            residual_correction = predict_hsgp_residual(residual_model, valid_frame)
            final_loss = residual_model["loss"]
            optimizer_success = residual_model["success"]
            optimizer_iterations = residual_model["nit"]
            n_basis = residual_model["n_basis"]

        pred_delta = valid_mean_delta + residual_correction
        pred_tvt = valid_last_known + pred_delta
        if not np.isfinite(pred_tvt).all():
            raise ValueError(f"Non-finite predictions for {spec['candidate']} fold {fold_idx}")

        metrics = metric_row(y_valid_tvt, pred_tvt)
        residual_sd = float(np.std(residual_correction))
        fold_rows.append({
            "fold": fold_idx,
            "candidate": spec["candidate"],
            "mean_model": spec["mean"],
            "coordinates": "+".join(spec["coordinates"]) if spec["coordinates"] else "none",
            "n_basis": n_basis,
            "n_train_rows": len(train_frame),
            "n_valid_rows": len(valid_frame),
            "final_loss": final_loss,
            "optimizer_success": optimizer_success,
            "optimizer_iterations": optimizer_iterations,
            "residual_correction_sd": residual_sd,
            **metrics,
        })
        posterior_rows.append({
            "fold": fold_idx,
            "candidate": spec["candidate"],
            "mean_model": spec["mean"],
            "coordinates": "+".join(spec["coordinates"]) if spec["coordinates"] else "none",
            "n_basis": n_basis,
            "final_loss": final_loss,
            "residual_correction_sd": residual_sd,
            "optimizer_success": optimizer_success,
            "optimizer_iterations": optimizer_iterations,
        })
        sample_idx = np.linspace(0, len(valid_frame) - 1, min(len(valid_frame), RESIDUAL_SAMPLE_PER_FOLD)).astype(int)
        residual_rows.append(pd.DataFrame({
            "fold": fold_idx,
            "candidate": spec["candidate"],
            "wellname": valid_frame.iloc[sample_idx]["wellname"].to_numpy(),
            "row_idx": valid_frame.iloc[sample_idx]["row_idx"].to_numpy(),
            "distance_from_boundary_md": valid_frame.iloc[sample_idx]["distance_from_boundary_md"].to_numpy(float),
            "eval_progress": valid_frame.iloc[sample_idx]["eval_progress"].to_numpy(float),
            "abs_projected_delta_10": valid_frame.iloc[sample_idx]["abs_projected_delta_10"].to_numpy(float),
            "y_true": y_valid_tvt[sample_idx],
            "mean_pred_tvt": valid_last_known[sample_idx] + valid_mean_delta[sample_idx],
            "y_pred": pred_tvt[sample_idx],
            "residual": y_valid_tvt[sample_idx] - pred_tvt[sample_idx],
            "residual_correction": residual_correction[sample_idx],
        }))

fit_seconds = time.time() - fit_start
fold_metrics = pd.DataFrame(fold_rows)
summary = (fold_metrics.groupby(["candidate", "mean_model", "coordinates", "n_basis"], as_index=False)
    .agg(rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"), mae_mean=("mae", "mean"), r2_mean=("r2", "mean"), residual_correction_sd_mean=("residual_correction_sd", "mean"), n_valid_rows=("n_valid_rows", "sum"), n_train_rows_mean=("n_train_rows", "mean"), final_loss_mean=("final_loss", "mean"), optimizer_success_rate=("optimizer_success", "mean"))
    .sort_values("rmse_mean"))
residual_sample = pd.concat(residual_rows, ignore_index=True)
posterior_diagnostics = pd.DataFrame(posterior_rows)

print({"fit_seconds": round(fit_seconds, 2), "fold_rows": len(fold_metrics), "residual_rows": len(residual_sample)})
summary

{'fit_seconds': 181.41, 'fold_rows': 18, 'residual_rows': 32400}


,candidate,mean_model,coordinates,n_basis,rmse_mean,rmse_std,mae_mean,r2_mean,residual_correction_sd_mean,n_valid_rows,n_train_rows_mean,final_loss_mean,optimizer_success_rate
5,huber_plus_hsgp_dps_m8,huber,distance_log1p+eval_progress+abs_projected_del...,24,14.358623,0.571308,10.079252,0.999462,1.351147,108220,72146.666667,23911.057074,1.0
3,huber_mean_only,huber,none,0,14.417152,0.549590,10.106583,0.999458,0.000000,108220,72146.666667,NaN,1.0
4,huber_plus_hsgp_dp_m8,huber,distance_log1p+eval_progress,16,14.423861,0.537204,10.109790,0.999458,0.577837,108220,72146.666667,24226.901445,1.0
2,gam_plus_hsgp_dps_m8,gam,distance_log1p+eval_progress+abs_projected_del...,24,46.065711,50.577619,12.727199,0.989539,0.627627,108220,72146.666667,25313.843443,1.0
1,gam_plus_hsgp_dp_m8,gam,distance_log1p+eval_progress,16,46.070000,50.545343,12.734095,0.989544,0.356956,108220,72146.666667,25376.704680,1.0
0,gam_mean_only,gam,none,0,46.090990,50.559087,12.733773,0.989536,0.000000,108220,72146.666667,NaN,1.0


## 5. Artifacts And Diagnostics

In [6]:
"Export residual-HSGP metrics, residual diagnostics, and figures."

fold_path = TABLE_DIR / f"model__hsgp_residual_on_gam_huber_fold_metrics__{RUN_DATE}.csv"
summary_path = TABLE_DIR / f"model__hsgp_residual_on_gam_huber_summary__{RUN_DATE}.csv"
posterior_path = TABLE_DIR / f"model__hsgp_residual_on_gam_huber_diagnostics__{RUN_DATE}.csv"
residual_path = TABLE_DIR / f"model__hsgp_residual_on_gam_huber_residual_sample__{RUN_DATE}.csv"
findings_path = TABLE_DIR / f"model__hsgp_residual_on_gam_huber_findings__{RUN_DATE}.csv"

fold_metrics.to_csv(fold_path, index=False)
summary.to_csv(summary_path, index=False)
posterior_diagnostics.to_csv(posterior_path, index=False)
residual_sample.to_csv(residual_path, index=False)

best = summary.iloc[0]
gam_mean = summary.loc[summary["candidate"] == "gam_mean_only"].iloc[0]
huber_mean = summary.loc[summary["candidate"] == "huber_mean_only"].iloc[0]
best_gam_hsgp = summary.loc[summary["candidate"].str.startswith("gam_plus")].iloc[0]
best_huber_hsgp = summary.loc[summary["candidate"].str.startswith("huber_plus")].iloc[0]
fine_tuned_gam_lb = 15.055
additive_hsgp_lb = 15.255

findings = pd.DataFrame([
    {"finding": "best_candidate", "value": str(best.candidate), "evidence": f"Best residual-HSGP sweep RMSE {best.rmse_mean:.4f}.", "implication": "Use this to decide whether to package a Kaggle notebook."},
    {"finding": "gam_hsgp_delta", "value": float(best_gam_hsgp.rmse_mean - gam_mean.rmse_mean), "evidence": f"Best GAM+HSGP {best_gam_hsgp.rmse_mean:.4f} vs GAM mean-only {gam_mean.rmse_mean:.4f}.", "implication": "Negative means residual HSGP improved the public-leading GAM direction locally."},
    {"finding": "huber_hsgp_delta", "value": float(best_huber_hsgp.rmse_mean - huber_mean.rmse_mean), "evidence": f"Best Huber+HSGP {best_huber_hsgp.rmse_mean:.4f} vs Huber mean-only {huber_mean.rmse_mean:.4f}.", "implication": "Negative means residual HSGP improved the locally strong Huber direction."},
    {"finding": "public_baseline_context", "value": fine_tuned_gam_lb, "evidence": f"Current best public LB is fine-tuned GAM {fine_tuned_gam_lb:.3f}; additive HSGP public LB is {additive_hsgp_lb:.3f}.", "implication": "Only package if local diagnostics show a plausible gain over GAM mean-only."},
])
findings.to_csv(findings_path, index=False)

fig_paths = {}

fig, ax = plt.subplots(figsize=(9, 5))
plot_df = summary.sort_values("rmse_mean", ascending=False)
ax.barh(plot_df["candidate"], plot_df["rmse_mean"], color="#4C78A8")
ax.set_xlabel("Grouped CV RMSE")
ax.set_title("HSGP Residual on GAM/Huber Means")
fig.tight_layout()
fig_paths["rmse_comparison"] = FIGURE_DIR / f"model__hsgp_residual_on_gam_huber_rmse_comparison__{RUN_DATE}.png"
fig.savefig(fig_paths["rmse_comparison"], dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 5))
for candidate, group in fold_metrics.groupby("candidate"):
    ax.plot(group["fold"], group["rmse"], marker="o", label=candidate)
ax.set_xlabel("Fold")
ax.set_ylabel("RMSE")
ax.set_title("Fold RMSE: Mean-only vs Residual HSGP")
ax.legend(fontsize=7, ncol=2)
fig.tight_layout()
fig_paths["fold_rmse"] = FIGURE_DIR / f"model__hsgp_residual_on_gam_huber_fold_rmse__{RUN_DATE}.png"
fig.savefig(fig_paths["fold_rmse"], dpi=160)
plt.close(fig)

best_residuals = residual_sample.loc[residual_sample["candidate"] == str(best.candidate)].copy()
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(best_residuals["distance_from_boundary_md"], best_residuals["residual_correction"], s=7, alpha=0.35, color="#54A24B")
ax.axhline(0.0, color="black", linewidth=1)
ax.set_title(f"Residual HSGP Correction vs Boundary Distance: {best.candidate}")
ax.set_xlabel("MD distance past boundary")
ax.set_ylabel("Residual correction")
fig.tight_layout()
fig_paths["correction_by_boundary"] = FIGURE_DIR / f"model__hsgp_residual_on_gam_huber_correction_boundary__{RUN_DATE}.png"
fig.savefig(fig_paths["correction_by_boundary"], dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(best_residuals["abs_projected_delta_10"], best_residuals["residual"], s=7, alpha=0.35, color="#E45756")
ax.axhline(0.0, color="black", linewidth=1)
ax.set_title(f"Residuals by Slope-Instability Coordinate: {best.candidate}")
ax.set_xlabel("abs projected delta 10")
ax.set_ylabel("TVT residual")
fig.tight_layout()
fig_paths["residual_by_slope_instability"] = FIGURE_DIR / f"model__hsgp_residual_on_gam_huber_residual_slope_instability__{RUN_DATE}.png"
fig.savefig(fig_paths["residual_by_slope_instability"], dpi=160)
plt.close(fig)

print({"tables": [str(fold_path), str(summary_path), str(posterior_path), str(residual_path), str(findings_path)], "figures": {k: str(v) for k, v in fig_paths.items()}})
findings

{'tables': ['C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\tables\\model__hsgp_residual_on_gam_huber_fold_metrics__2026-06-22.csv', 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\tables\\model__hsgp_residual_on_gam_huber_summary__2026-06-22.csv', 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\tables\\model__hsgp_residual_on_gam_huber_diagnostics__2026-06-22.csv', 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\tables\\model__hsgp_residual_on_gam_huber_residual_sample__2026-06-22.csv', 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\tables\\model__hsgp_residual_on_gam_huber_findings__2026-06-22.csv'], 'figures': {'rmse_comparison': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__hsgp_residual_on_gam_huber_rmse_comp

,finding,value,evidence,implication
0,best_candidate,huber_plus_hsgp_dps_m8,Best residual-HSGP sweep RMSE 14.3586.,Use this to decide whether to package a Kaggle...
1,gam_hsgp_delta,-0.025279,Best GAM+HSGP 46.0657 vs GAM mean-only 46.0910.,Negative means residual HSGP improved the publ...
2,huber_hsgp_delta,-0.058529,Best Huber+HSGP 14.3586 vs Huber mean-only 14....,Negative means residual HSGP improved the loca...
3,public_baseline_context,15.055,Current best public LB is fine-tuned GAM 15.05...,Only package if local diagnostics show a plaus...


In [7]:
"Write report, update model comparison, and create run bundle."

best = summary.iloc[0]
gam_mean = summary.loc[summary["candidate"] == "gam_mean_only"].iloc[0]
huber_mean = summary.loc[summary["candidate"] == "huber_mean_only"].iloc[0]
best_gam_hsgp = summary.loc[summary["candidate"].str.startswith("gam_plus")].iloc[0]
best_huber_hsgp = summary.loc[summary["candidate"].str.startswith("huber_plus")].iloc[0]

report_path = REPORT_DIR / f"model__hsgp_residual_on_gam_huber__{RUN_DATE}.md"
report_lines = [
    "# HSGP Residuals on GAM and Huber Means",
    "",
    f"Date: {RUN_DATE}",
    "",
    "## Key Findings",
]
for row in findings.itertuples(index=False):
    report_lines.extend([
        f"- **{row.finding}**: {row.evidence}",
        f"  - Modeling implication: {row.implication}",
    ])
report_lines.extend([
    "",
    "## Best Candidate",
    f"- Candidate: `{best.candidate}`",
    f"- Mean model: `{best.mean_model}`",
    f"- Coordinates: `{best.coordinates}`",
    f"- RMSE mean/std: {best.rmse_mean:.4f} / {best.rmse_std:.4f}",
    f"- GAM residual-HSGP delta vs GAM mean-only: {best_gam_hsgp.rmse_mean - gam_mean.rmse_mean:.4f}",
    f"- Huber residual-HSGP delta vs Huber mean-only: {best_huber_hsgp.rmse_mean - huber_mean.rmse_mean:.4f}",
    "",
    "## Artifacts",
    f"- Fold metrics: {fold_path}",
    f"- Summary: {summary_path}",
    f"- Diagnostics: {posterior_path}",
    f"- Residual sample: {residual_path}",
    f"- Findings: {findings_path}",
])
for name, path in fig_paths.items():
    report_lines.append(f"- {name}: {path}")
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

comparison_path = TABLE_DIR / "model_comparison.csv"
comparison_run_id = "2026-06-22__hsgp_residual_on_gam_huber"
comparison_row = pd.DataFrame([{
    "run_id": comparison_run_id,
    "model_family": "bayesian_hsgp_residual_mean",
    "model_name": str(best.candidate),
    "cv_rmse_mean": float(best.rmse_mean),
    "cv_rmse_std": float(best.rmse_std),
    "train_time_s": float(fit_seconds),
    "n_features": int(best.n_basis),
    "notes": "Student-t MAP additive Hilbert residual on top of tuned GAM/Huber means",
}])
if comparison_path.exists():
    comparison = pd.read_csv(comparison_path)
    comparison = comparison.loc[comparison["run_id"] != comparison_run_id].copy()
    comparison = pd.concat([comparison, comparison_row], ignore_index=True)
else:
    comparison = comparison_row
comparison.to_csv(comparison_path, index=False)

config = {
    "task": RUN_TAG,
    "seed": SEED,
    "n_splits": N_SPLITS,
    "max_target_rows_per_well": MAX_TARGET_ROWS_PER_WELL,
    "max_train_rows_per_fold": MAX_TRAIN_ROWS_PER_FOLD,
    "hsgp_m": HSGP_M,
    "hsgp_ell": HSGP_ELL,
    "hsgp_coordinates": HSGP_COORDINATES,
    "gam": {"n_knots": GAM_N_KNOTS, "degree": GAM_DEGREE, "alpha": GAM_ALPHA},
    "huber": {"alpha": HUBER_ALPHA, "epsilon": HUBER_EPSILON},
    "best_candidate": str(best.candidate),
}
with RunBundle(RUN_TAG, config, seed=SEED) as rb:
    rb.add_input(str(TRAIN_DIR))
    rb.add_metric("sample_rows", float(len(feature_frame)))
    rb.add_metric("best_rmse", float(best.rmse_mean))
    rb.add_metric("gam_hsgp_delta", float(best_gam_hsgp.rmse_mean - gam_mean.rmse_mean))
    rb.add_metric("huber_hsgp_delta", float(best_huber_hsgp.rmse_mean - huber_mean.rmse_mean))
    rb.add_output(str(fold_path))
    rb.add_output(str(summary_path))
    rb.add_output(str(posterior_path))
    rb.add_output(str(residual_path))
    rb.add_output(str(findings_path))
    rb.add_output(str(report_path))
    for path in fig_paths.values():
        rb.add_output(str(path))
    rb.set_notes("Residual HSGP test on top of the current best GAM public direction and strongest local Huber direction.")

print(report_path)
summary

C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\reports\model__hsgp_residual_on_gam_huber__2026-06-22.md


,candidate,mean_model,coordinates,n_basis,rmse_mean,rmse_std,mae_mean,r2_mean,residual_correction_sd_mean,n_valid_rows,n_train_rows_mean,final_loss_mean,optimizer_success_rate
5,huber_plus_hsgp_dps_m8,huber,distance_log1p+eval_progress+abs_projected_del...,24,14.358623,0.571308,10.079252,0.999462,1.351147,108220,72146.666667,23911.057074,1.0
3,huber_mean_only,huber,none,0,14.417152,0.549590,10.106583,0.999458,0.000000,108220,72146.666667,NaN,1.0
4,huber_plus_hsgp_dp_m8,huber,distance_log1p+eval_progress,16,14.423861,0.537204,10.109790,0.999458,0.577837,108220,72146.666667,24226.901445,1.0
2,gam_plus_hsgp_dps_m8,gam,distance_log1p+eval_progress+abs_projected_del...,24,46.065711,50.577619,12.727199,0.989539,0.627627,108220,72146.666667,25313.843443,1.0
1,gam_plus_hsgp_dp_m8,gam,distance_log1p+eval_progress,16,46.070000,50.545343,12.734095,0.989544,0.356956,108220,72146.666667,25376.704680,1.0
0,gam_mean_only,gam,none,0,46.090990,50.559087,12.733773,0.989536,0.000000,108220,72146.666667,NaN,1.0


## Results

Review the summary table above and the generated report for promotion decisions.